# Using ESP32 (or Arduino) PWM for a Function Generator and White-Noise Generator

In this project, we will build a `SW DAC` using 100 kHz PWM and an RC filter. This `SW DAC` will be used to generate simple waveforms and to create a white-noise sound generator.

In [1]:
import pandas as pd 
import numpy as np

In [2]:
# CONFIGURATIONS
#help(handcalcs.set_option)
import handcalcs.render
handcalcs.set_option("latex_block_start", "$")
handcalcs.set_option("latex_block_end", "$")
handcalcs.set_option("math_environment_end", "aligned")
handcalcs.set_option("use_scientific_notation",True)

In [3]:
import os

# Get current working directory
cwd = os.getcwd()
prin= 0
if prin: 
    print("Current working directory:", cwd)


## Setup

In [4]:
from localcode import *
fig_counter=1

Below the setup used to check the DAC idea

In [5]:
first_part_path = r"../05_images/"
fig_counter= One_figure_with_captions(
    fig_counter=fig_counter, 
     img1="setup.png", 
     caption1="The test setup",  
     first_part_path=first_part_path, 
     width=9)

The RC smooths the PWM to acte like a mouving average, the potentiometer is only used to adjust the peak-to-peak level. 
    FYI: The TDS oscillo has an input impedance of $1M\ohm$ and $13pF$.

In [6]:
from math import pi , sqrt, log , exp, log10, sin

In [7]:
%%render 
R = 460 # ohm
C = 1 # µF
To = R*C # µs 
F_cutoff = 1e6/(To *2*pi) # Hz 

<IPython.core.display.Latex object>

## Test results

### Simple wave generator

**C++ code of the ESP32**

```c
int pwmPin = 26; // GPIO26

void setup() {
  ledcAttach(pwmPin, 100000, 8); // Attach GPIO26 to LEDC and configure it 100kHz
}

void loop() {
  for (int duty = 0; duty <= 255; duty = duty + 10) {
    ledcWrite(pwmPin, duty); // ledcWrite now takes the pin number
    delay(1);
  }
}
```

**Oscilloscope screenshots**

In [8]:
first_part_path = r"../01_Simple_DAC/"
fig_counter= One_figure_with_captions(
    fig_counter=fig_counter, 
     img1="Oscillo.png", 
     caption1="Sawtooth waveform, Potentiometer = 100%",  
     first_part_path=first_part_path, 
     width=9)

The response rate is limited because of the RC circuit’s time constant.

### 50 Hz sine wave generator

**C++ code of the ESP32**

```c
#include <cmath> // Include the cmath library for sin() and M_PI

int pwmPin = 26; // GPIO26

void setup() {
  ledcAttach(pwmPin, 100000, 8); // Attach GPIO26 to LEDC and configure it 100kHz
}

const int freq = 50; 
const float Tper_us = 1000000.0 / freq; 

void loop() {
  // Get the current time in microseconds within a single period
  unsigned long time_in_period = micros() % (unsigned long)Tper_us;
  
  // Convert the time to a normalized angle from 0 to 2*PI radians
  // The expression is (time_in_period / Tper_us) * (2 * M_PI)
  float angle = (float)time_in_period / Tper_us * 2.0 * M_PI;

  // Calculate the duty cycle from the sine wave
  // The value will range from 0 to 255 (for 8-bit resolution)
  int duty = (int)(127.5 * (sin(angle) + 1.0));

  ledcWrite(pwmPin, duty);
}
```

**Oscilloscope screenshots**

In [9]:
first_part_path = r"../02_sin_wave_50Hz/"
fig_counter = show_figure_pair_with_captions(
    fig_counter, 
    img1= "Oscillo1.png", 
    caption1= "Very clean sine wave, Potentiometer = 100%", 
    img2="Oscillo2_pententiometre_around50.png", 
    caption2= "Potentiometer ≈ 50%", 
    first_part_path=first_part_path)

Since 50 Hz is relatively slow, the output is very clean.

We can estimate the attenuation of the 100 kHz harmonic using the simplified formula:

In [10]:
%%render 
Fsw = 100e3 # Hz 
F_cutoff  # Hz 
F_rate = Fsw/F_cutoff
Nb_decades = log10(F_rate)

<IPython.core.display.Latex object>

Since we have first order low pass filter : Attenuation slop is  –20 dB/decade

In [11]:
%%render long 
F_SW_Attenuation= -20*Nb_decades # dB
F_SW_LinearAttenuation = 10**(F_SW_Attenuation/20)

<IPython.core.display.Latex object>

so 1 V of PWM is reduced to ~3mV after the RC filter

### White noise generator

**C++ code of the ESP32**

A fast custom random function `fastRandom()` is implemented here to replace the standard `rand()`, which is too slow for high-frequency updates.

```c
int pwmPin = 26; // GPIO26


/////////// RANDOM FAST FUNCTION 
uint32_t seed = 123456789;
uint8_t fastRandom() {
    seed = (1664525UL * seed + 1013904223UL);
    return (seed >> 24) & 0xFF;  // returns 0–255
}
/////////////////////////////


int delay_us = 100; //10kHz of update 
void setup() {
  ledcAttach(pwmPin, 100000, 8); // Attach GPIO26 to LEDC and configure it 100kHz
}

void loop() {
  int duty = fastRandom() % 256;
    ledcWrite(pwmPin, duty); // ledcWrite now takes the pin number
    delayMicroseconds(delay_us); 
  
}
```

**Oscilloscope screenshots**

In [12]:
first_part_path = r"../03_white_noise/"
fig_counter = show_figure_pair_with_captions(
    fig_counter, 
    img1= "1_update_2kHz.jpg", 
    caption1= "Noise form, 2kHz speed update", 
    img2="1_update_2kHz_fft.png", 
    caption2= "Noise form, 2kHz speed update, oscilloscope FFT", 
    first_part_path=first_part_path)

fig_counter = show_figure_pair_with_captions(
    fig_counter, 
     img1="1_update_500Hz_fft.png", 
     caption1="Noise form and oscilloscope FFT, 500Hz speed update", 
    img2="FFT500Hz1kHz.png", 
    caption2= "Python FFT of 500Hz vs 2kHz", 
    first_part_path=first_part_path)



The FFT of the pseudo white noise is not perfectly uniform, but it is very rich in frequency content.

## Bonus: implement a simple white-noise generator for children.

In this section, I will show how to quickly build a white-noise sound generator using an ESP32 and a small Bluetooth audio amplifier. The PCB is modified to route the ESP32’s audio output to the amplifier in place of the original MCU output. Below is the simplified schematic of this device.

In [13]:
imgs = [
     
    "audio_.png", 
    "photo (2).jpg", 
    "photo (3).jpg", 
    "photo (1).jpg"
]

captions= [
  
    "Amplifier PCB", 
    "Amplifier PCB with ESP32", 
    "Final result 1", 
    "Final result 2"
]

In [14]:
first_part_path = r"../05_images/"
fig_counter= One_figure_with_captions(
    fig_counter=fig_counter, 
     img1="audio_schematic.png", 
     caption1="Proposed schematic",  
     first_part_path=first_part_path, 
     width=17)

The `8002D` is isolated on the PCB and powered directly with 5V as you can see in the schematic above.
<br>
Below some images of the final result of the white noise generator

[You can download the 8002D CF4F1K audio amplifier datasheet here](https://datasheet4u.com/datasheet/ChipSourceTek/8002D-1541828)


In [15]:



for i in  range(0, len(imgs)-1, 2):
    fig_counter = show_figure_pair_with_captions(
        fig_counter, 
        img1= imgs[i], 
        caption1= captions[i], 
        img2=imgs[i+1], 
        caption2=captions[i+1], 
        first_part_path=first_part_path)


The results are satisfactory, and the potentiometer works well to adjust the sound level.

Below is a phone recording of the sound. The quality is not very accurate, but it’s enough to check the result.

In [16]:
import base64
from IPython.display import HTML

mp3_path = "../05_images/RECORDED_Sound.mp3"  # adjust if needed
mp3_data = base64.b64encode(open(mp3_path,"rb").read()).decode()

HTML(f"""
<audio controls>
  <source src="data:audio/mpeg;base64,{mp3_data}" type="audio/mpeg">
</audio>
""")
